# estimate

Estimate GPU-hours for a full pipeline run based on calibration results.

Usage:
    uv run estimate-calibration [N_ADS]

Arguments:
    N_ADS     Total ads to estimate for (default: actual count from adzuna.duckdb)

Reads calibration results from store/calibration/results/ and estimates
per-node GPU-hours at scale.

In [ ]:
#|default_exp estimate

In [ ]:
#|export
import json
import sys
from pathlib import Path

from ai_index.const import calibration_results_path

RESULTS_DIR = calibration_results_path

# Nodes with per-ad scaling
PER_AD_NODES = ["embed_ads", "cosine_candidates", "llm_filter_candidates", "rerank_candidates"]
# Nodes with fixed cost (independent of n_ads)
FIXED_NODES = ["embed_onet"]

In [ ]:
#|export
def _count_ads() -> int:
    """Count total ads in the Adzuna DuckDB database."""
    import duckdb
    from ai_index.const import inputs_path
    con = duckdb.connect(str(inputs_path / "adzuna.duckdb"), read_only=True)
    count = con.sql("SELECT COUNT(*) FROM ads").fetchone()[0]
    con.close()
    return count

In [ ]:
#|export
def _load_results() -> list[dict]:
    results = []
    if RESULTS_DIR.exists():
        for path in sorted(RESULTS_DIR.glob("*.json")):
            with open(path) as f:
                results.append(json.load(f))
    return results

In [ ]:
#|export
def main():
    n_ads = int(sys.argv[1]) if len(sys.argv) > 1 else _count_ads()

    results = _load_results()
    if not results:
        print(f"No calibration results found in {RESULTS_DIR}/")
        print("Run: uv run run-calibration <llm_model> <embed_model> [<rerank_model>]")
        sys.exit(1)

    print(f"\nGPU-hour estimates for {n_ads:,} ads")
    print("=" * 100)

    for r in results:
        models = f"embed={r['embedding_model']}, llm={r['llm_model']}"
        if r.get("rerank_model"):
            models += f", rerank={r['rerank_model']}"
        print(f"\n{models}")
        print(f"  Calibrated on {r['sample_n']:,} ads at {r['timestamp']}")
        print(f"  {'Node':30s} {'s/ad':>8} {'Est. hours':>12} {'NHR':>8} {'Source':>8}")
        print(f"  {'-'*76}")

        nodes = r["nodes"]
        total_hours = 0.0

        for node_name in PER_AD_NODES:
            if node_name not in nodes:
                print(f"  {node_name:30s} {'n/a':>8}")
                continue
            timing = nodes[node_name]
            spa = timing["seconds_per_ad"]
            hours = spa * n_ads / 3600
            source = "slurm" if timing.get("slurm_seconds") else "clock"
            nhr = hours * 0.25
            total_hours += hours
            print(f"  {node_name:30s} {spa:>8.4f} {hours:>12.1f} {nhr:>8.1f} {source:>8}")

        for node_name in FIXED_NODES:
            if node_name not in nodes:
                continue
            timing = nodes[node_name]
            hours = timing["elapsed_seconds"] / 3600
            source = "slurm" if timing.get("slurm_seconds") else "clock"
            nhr = hours * 0.25
            total_hours += hours
            print(f"  {node_name:30s} {'fixed':>8} {hours:>12.3f} {nhr:>8.3f} {source:>8}")

        total_nhr = total_hours * 0.25
        print(f"  {'TOTAL':30s} {'':>8} {total_hours:>12.1f} {total_nhr:>8.1f}")